## Modelación

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

import joblib

sns.set_theme(style="white")

### 0. Carga de datos

In [2]:
# Ruta de la carpeta donde se guardaron los datos
RUTA_DATOS_MODELO = Path("../datos_modelo")


# Cargamos las características
X_train = np.load(
    RUTA_DATOS_MODELO / "X_train.npy",
    allow_pickle=False
)

X_test = np.load(
    RUTA_DATOS_MODELO / "X_test.npy",
    allow_pickle=False
)


# Cargamos las etiquetas
y_train = np.load(
    RUTA_DATOS_MODELO / "y_train.npy",
    allow_pickle=False
)

y_test = np.load(
    RUTA_DATOS_MODELO / "y_test.npy",
    allow_pickle=False
)


# Cargamos los nombres de las características
nombres_caracteristicas = np.load(
    RUTA_DATOS_MODELO / "nombres_caracteristicas.npy",
    allow_pickle=True
)


# Mostramos las dimensiones para comprobar la carga
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print(
    "Nombres de características:",
    nombres_caracteristicas.shape
)

X_train: (12448, 120)
y_train: (12448,)
X_test: (3112, 120)
y_test: (3112,)
Nombres de características: (120,)


## 2. Definición de los modelos

Se compararán tres algoritmos de clasificación utilizando los mismos conjuntos de entrenamiento y prueba:

1. **Regresión logística:** establece relaciones lineales entre las características y las actividades.
2. **SVM:** busca límites de separación entre las actividades y puede representar relaciones no lineales.
3. **Random Forest:** combina varios árboles de decisión y puede reconocer relaciones más complejas entre las características.

La regresión logística y SVM requieren que sus características tengan escalas comparables. Por esta razón, su estandarización se incluirá dentro de un pipeline. Random Forest no necesita estandarización.

Los tres modelos generarán probabilidades para cada ventana. Estas probabilidades se utilizarán posteriormente para obtener una sola predicción por señal completa.

In [3]:
# ---------------------------------------------------------
# Modelo 1: Regresión logística
# ---------------------------------------------------------

modelo_logistico = Pipeline([
    (
        "escalado",
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])


# ---------------------------------------------------------
# Modelo 2: Máquina de soporte vectorial
# ---------------------------------------------------------

modelo_svm = Pipeline([
    (
        "escalado",
        StandardScaler()
    ),
    (
        "modelo",
        SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        )
    )
])


# ---------------------------------------------------------
# Modelo 3: Random Forest
# ---------------------------------------------------------

modelo_random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


# Diccionario para recorrer los tres modelos
modelos = {
    "Regresión logística": modelo_logistico,
    "SVM": modelo_svm,
    "Random Forest": modelo_random_forest
}

print("Modelos preparados:")
for nombre in modelos:
    print("-", nombre)

Modelos preparados:
- Regresión logística
- SVM
- Random Forest


## 3. Combinación de ventanas por señal completa

Cada señal original está formada por cuatro ventanas y el modelo genera una distribución de probabilidades para cada una. Para obtener una sola clasificación por señal completa, se promediarán las probabilidades de sus cuatro ventanas.

Este método es preferible a la votación mayoritaria simple porque considera la seguridad de cada predicción. También permite resolver casos en los que dos ventanas predicen una actividad y las otras dos predicen otra, sin seleccionar arbitrariamente una de las clases empatadas.

La actividad con la probabilidad promedio más alta será la predicción final de la señal completa. Los metadatos se utilizarán exclusivamente para agrupar las ventanas mediante `sample_id` y no entrarán como características del modelo.

In [4]:
# ---------------------------------------------------------
# Carga de los metadatos de prueba
# ---------------------------------------------------------

metadatos_test = pd.read_csv(
    RUTA_DATOS_MODELO / "metadatos_test.csv",
    dtype={
        "sample_id": str,
        "actividad": str
    }
)

if len(metadatos_test) != len(X_test):
    raise ValueError(
        "La cantidad de metadatos no coincide con X_test."
    )

In [5]:
# ---------------------------------------------------------
# Función para combinar probabilidades por señal
# ---------------------------------------------------------

def predecir_senales_completas(
    modelo,
    X_prueba,
    y_prueba,
    metadatos
):
    """
    Promedia las probabilidades de las cuatro ventanas
    correspondientes a cada señal completa.
    """

    # Probabilidades de cada actividad para cada ventana.
    probabilidades = modelo.predict_proba(
        X_prueba
    )

    # Las clases indican a qué actividad corresponde
    # cada columna de probabilidades.
    clases = modelo.classes_

    # Cuando se utiliza un Pipeline, las clases pertenecen
    # al último paso, es decir, al clasificador.
    if isinstance(modelo, Pipeline):
        clases = modelo.named_steps["modelo"].classes_

    resultados = []

    # Agrupamos las ventanas mediante sample_id.
    for sample_id, indices in metadatos.groupby(
        "sample_id",
        sort=False
    ).groups.items():

        # Posiciones de las cuatro ventanas de la señal.
        indices = np.asarray(
            list(indices)
        )

        # Promedio de probabilidades de las cuatro ventanas.
        probabilidad_promedio = np.mean(
            probabilidades[indices],
            axis=0
        )

        # Posición de la actividad con mayor probabilidad.
        indice_ganador = np.argmax(
            probabilidad_promedio
        )

        actividad_predicha = clases[
            indice_ganador
        ]

        # Todas las ventanas de la señal tienen la misma
        # actividad real, por eso tomamos la primera.
        actividad_real = y_prueba[
            indices[0]
        ]

        resultados.append({
            "sample_id": sample_id,
            "actividad_real": actividad_real,
            "actividad_predicha": actividad_predicha,
            "probabilidad_maxima": np.max(
                probabilidad_promedio
            ),
            "cantidad_ventanas": len(indices)
        })

    return pd.DataFrame(resultados)

## 4. Entrenamiento y evaluación

Los tres modelos se entrenarán utilizando `X_train` y `y_train`. Después, cada modelo clasificará las ventanas de prueba y se calculará su exactitud por ventana.

Finalmente, las probabilidades de las cuatro ventanas se combinarán para calcular la exactitud y el F1-score macro por señal completa. El F1-score macro asigna la misma importancia a cada una de las 16 actividades.

In [6]:
# ---------------------------------------------------------
# Entrenamiento y comparación
# ---------------------------------------------------------

resultados_modelos = []
predicciones_senales = {}

for nombre, modelo in modelos.items():

    print(f"\nEntrenando: {nombre}")

    # Entrenamiento
    modelo.fit(
        X_train,
        y_train
    )

    # Predicción individual de las ventanas
    y_pred_ventanas = modelo.predict(
        X_test
    )

    exactitud_ventanas = accuracy_score(
        y_test,
        y_pred_ventanas
    )

    # Predicción de las señales completas mediante
    # promedio de probabilidades.
    resultados_senal = predecir_senales_completas(
        modelo,
        X_test,
        y_test,
        metadatos_test
    )

    y_real_senal = resultados_senal[
        "actividad_real"
    ]

    y_pred_senal = resultados_senal[
        "actividad_predicha"
    ]

    exactitud_senal = accuracy_score(
        y_real_senal,
        y_pred_senal
    )

    f1_macro_senal = f1_score(
        y_real_senal,
        y_pred_senal,
        average="macro"
    )

    resultados_modelos.append({
        "modelo": nombre,
        "exactitud_ventanas": exactitud_ventanas,
        "exactitud_senal": exactitud_senal,
        "f1_macro_senal": f1_macro_senal
    })

    # Guardamos las predicciones para analizarlas después.
    predicciones_senales[nombre] = resultados_senal

    print(
        f"Exactitud por ventana: "
        f"{exactitud_ventanas:.4f}"
    )

    print(
        f"Exactitud por señal: "
        f"{exactitud_senal:.4f}"
    )

    print(
        f"F1-score macro por señal: "
        f"{f1_macro_senal:.4f}"
    )


# Convertimos el resumen en una tabla.
tabla_resultados = pd.DataFrame(
    resultados_modelos
)

display(tabla_resultados)


Entrenando: Regresión logística
Exactitud por ventana: 0.7866
Exactitud por señal: 0.8689
F1-score macro por señal: 0.8589

Entrenando: SVM
Exactitud por ventana: 0.8869
Exactitud por señal: 0.9434
F1-score macro por señal: 0.9369

Entrenando: Random Forest
Exactitud por ventana: 0.9569
Exactitud por señal: 0.9756
F1-score macro por señal: 0.9714


,modelo,exactitud_ventanas,exactitud_senal,f1_macro_senal
0,Regresión logística,0.786632,0.868895,0.858903
1,SVM,0.886889,0.943445,0.936919
2,Random Forest,0.956941,0.975578,0.971376


### Interpretación de la comparación

Los tres modelos fueron evaluados utilizando las mismas ventanas de prueba y el mismo procedimiento para obtener una predicción por señal completa. En todos los casos, la exactitud por señal fue mayor que la exactitud por ventana, lo que indica que promediar las probabilidades de las cuatro ventanas permite obtener una clasificación más estable.

La **regresión logística** obtuvo una exactitud de **78.66% por ventana** y **86.89% por señal completa**. Su F1-score macro por señal fue de **0.8589**. Este fue el desempeño más bajo de los tres modelos, lo que sugiere que las actividades no pueden separarse adecuadamente utilizando únicamente relaciones lineales entre las características.

El modelo **SVM** obtuvo una exactitud de **88.69% por ventana** y **94.34% por señal completa**, con un F1-score macro de **0.9369**. La mejora respecto a la regresión logística indica que el kernel no lineal permitió representar mejor las diferencias entre las actividades.

El modelo **Random Forest** obtuvo el mejor resultado, con una exactitud de **95.69% por ventana** y **97.56% por señal completa**. Su F1-score macro fue de **0.9714**, lo que muestra que el desempeño se mantuvo alto al asignar la misma importancia a las 16 actividades.

La cercanía entre la exactitud y el F1-score macro de Random Forest indica que su buen resultado no se debe únicamente a las actividades con mayor cantidad de señales. Con esta configuración inicial, Random Forest es el modelo con mejor capacidad para clasificar las actividades del conjunto de datos.